In [1]:
import torch
import numpy as np
import nibabel as nib
from pathlib import Path
from typing import Any
from models import get_model
from utils import parse_cfg
from datasets import create_input_space, create_input_space_prop, create_input_space_prop_upsampled
from tqdm import tqdm
import time
from datetime import datetime

In [3]:
def upsample_efficient(
    cfg_path: Path,
    checkpoint_path: Path,
    train_dims: tuple[int, int, int],
    pred_dims: tuple[int, int, int],
    reference_nifti_path: Path,
    save_path: Path,
    batch_size: int = 10000,
    multishell: bool = False
):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    # Load config & model
    cfg = parse_cfg(cfg_path)
    model = get_model(cfg).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    # Create high-res input space
    input_space = create_input_space_prop_upsampled(
        *train_dims, *pred_dims
    ).reshape(-1, 3).to(device)

    print(f"Input space shape: {input_space.shape}")

    # Output shape
    output_channels = 45
    output_shape = (*pred_dims, output_channels)

    # Create memory-mapped file to avoid RAM issues
    output_memmap_path = save_path.with_suffix(".npy")
    output_mm = np.memmap(output_memmap_path, dtype=np.float32, mode='w+', shape=output_shape)

    # Flatten voxel index order for tracking
    total_voxels = input_space.shape[0]
    spatial_indices = torch.arange(total_voxels)

    # Inference loop, batch-wise, streaming to memmap
    with torch.no_grad():
        for start in tqdm(range(0, total_voxels, batch_size), desc="Running inference"):
            end = min(start + batch_size, total_voxels)
            batch = input_space[start:end]
            out = model(batch)
            if multishell:
                out = out[..., :-2]
            out = out.cpu().numpy()

            # Write predictions to correct spatial locations
            flat_indices = spatial_indices[start:end].cpu().numpy()
            x, y, z = np.unravel_index(flat_indices, pred_dims)  # (x, y, z)
            output_mm[x, y, z, :] = out

    output_mm.flush()

    # Prepare affine matrix from reference image
    ref_img = nib.load(str(reference_nifti_path))
    original_affine = ref_img.affine.copy()
    scale_factors = [old / new for new, old in zip(pred_dims, train_dims)]

    scale_mat = np.array([
        [scale_factors[0], 0, 0, 0],
        [0, scale_factors[1], 0, 0],
        [0, 0, scale_factors[2], 0],
        [0, 0, 0, 1]
    ])

    # Save directly from memmap without loading into RAM
    nifti_img = nib.Nifti2Image(output_mm, original_affine * scale_mat)
    nib.save(nifti_img, str(save_path))

In [4]:
### EXAMPLE RUN

# === Define paths ===
cfg_path = Path('configs/sub-CON02_config_multi.yaml')
cfg = parse_cfg(cfg_path)
checkpoint_path = Path('runs/sub-CON02/model_prop.pt')
reference_nifti_path = Path('runs/sub-CON02/coeffs_prop.nii.gz')
save_path = Path("/sub-CON02/x10/INR/coeffs_x10.nii.gz")

# === Define resolutions ===
scale_factor = 10
train_dims = (cfg['width'], cfg['height'], cfg['depth'])   # resolution used during training
pred_dims = tuple(dim * scale_factor for dim in train_dims)      # desired super-res output resolution

upsample_efficient(
    cfg_path=cfg_path,
    checkpoint_path=checkpoint_path,
    train_dims=train_dims,
    pred_dims=pred_dims,
    reference_nifti_path=reference_nifti_path,
    save_path=save_path,
    batch_size=1000,
    multishell=True,
)